# 복지서비스 데이터 수집 (data.go.kr)

한국사회보장정보원 Open API 2종에서 데이터를 받아 pandas DataFrame으로 봅니다.

| 구분 | 서비스 | 목록조회 | 상세조회 |
|---|---|---|---|
| 중앙부처 | `NationalWelfareInformationsV001` | `NationalWelfarelistV001` | `NationalWelfaredetailedV001` |
| 지자체 | `LocalGovernmentWelfareInformations` | `LcgvWelfarelist` | `LcgvWelfaredetailed` |

추가로 행정안전부 **정부24 공공서비스 정보** API(`api.odcloud.kr/api/gov24/v3`)도 받습니다 (7장).

인증키는 같은 폴더의 `.env` 파일에서 `DATA_GO_KR_KEY` 값을 읽습니다.

In [ ]:
# 필요 시 한 번만 실행
# %pip install requests pandas

In [ ]:
import html
import os
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


def load_env(path=".env"):
    p = Path(path)
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())


load_env()
SERVICE_KEY = os.environ["DATA_GO_KR_KEY"]

APIS = {
    "중앙부처": {
        "base": "https://apis.data.go.kr/B554287/NationalWelfareInformationsV001",
        "list": "NationalWelfarelistV001",
        "detail": "NationalWelfaredetailedV001",
        # 중앙부처 API는 callTp, srchKeyCode가 필수
        "list_params": {"callTp": "L", "srchKeyCode": "003"},
        "detail_params": {"callTp": "D"},
    },
    "지자체": {
        "base": "https://apis.data.go.kr/B554287/LocalGovernmentWelfareInformations",
        "list": "LcgvWelfarelist",
        "detail": "LcgvWelfaredetailed",
        "list_params": {},
        "detail_params": {},
    },
}

In [ ]:
def _get_xml(url, params):
    res = requests.get(url, params={"serviceKey": SERVICE_KEY, **params}, timeout=30)
    res.raise_for_status()
    try:
        root = ET.fromstring(res.content)
    except ET.ParseError:
        raise RuntimeError(f"XML 파싱 실패: {res.text[:300]}")
    code = root.findtext("resultCode")
    if code not in (None, "0"):
        raise RuntimeError(f"API 오류 {code}: {root.findtext('resultMessage')}")
    # 인증키 오류 등은 OpenAPI_ServiceResponse 형태로 옴
    err = root.findtext(".//returnAuthMsg")
    if err:
        raise RuntimeError(f"API 오류: {err}")
    return root


def _clean(text):
    # 줄바꿈(\r) 정리 + 이중 이스케이프된 문자(&#9312; 등) 복원
    return html.unescape((text or "").replace("\r", "\n")).strip()


def _elem_to_dict(elem):
    return {child.tag: _clean(child.text) for child in elem}


def fetch_list_page(api="중앙부처", page_no=1, num_of_rows=500, **filters):
    """목록조회 한 페이지 → (전체건수, DataFrame)"""
    cfg = APIS[api]
    params = {**cfg["list_params"], "pageNo": page_no, "numOfRows": num_of_rows, **filters}
    root = _get_xml(f"{cfg['base']}/{cfg['list']}", params)
    total = int(root.findtext("totalCount") or 0)
    rows = [_elem_to_dict(e) for e in root.findall("servList")]
    return total, pd.DataFrame(rows)


def fetch_all_list(api="중앙부처", num_of_rows=500, sleep=0.2, **filters):
    """모든 페이지를 돌며 목록 전체를 DataFrame으로 반환

    filters 예: searchWrd="청년", orderBy="popular"
    """
    total, df = fetch_list_page(api, 1, num_of_rows, **filters)
    frames = [df]
    pages = (total + num_of_rows - 1) // num_of_rows
    for page in range(2, pages + 1):
        time.sleep(sleep)
        _, df = fetch_list_page(api, page, num_of_rows, **filters)
        frames.append(df)
    result = pd.concat(frames, ignore_index=True)
    print(f"[{api}] 전체 {total}건 중 {len(result)}건 수집")
    return result


def fetch_detail(serv_id, api="중앙부처"):
    """상세조회 → (기본정보 dict, {목록이름: DataFrame})

    신청방법, 문의처, 근거법령 같은 반복 항목은 DataFrame으로 따로 묶어서 돌려줍니다.
    """
    cfg = APIS[api]
    root = _get_xml(f"{cfg['base']}/{cfg['detail']}", {**cfg["detail_params"], "servId": serv_id})
    info, lists = {}, {}
    for child in root:
        if child.tag in ("resultCode", "resultMessage"):
            continue
        if len(child):  # 하위 요소가 있으면 반복 목록
            lists.setdefault(child.tag, []).append(_elem_to_dict(child))
        else:
            info[child.tag] = _clean(child.text)
    return info, {k: pd.DataFrame(v) for k, v in lists.items()}

## 1. 중앙부처 복지서비스 목록

In [ ]:
df_nat = fetch_all_list("중앙부처")
df_nat.head()

## 2. 지자체 복지서비스 목록

In [ ]:
df_loc = fetch_all_list("지자체")
df_loc.head()

## 3. 두 데이터 합치기

두 API는 같은 의미의 컬럼 이름이 다릅니다 (예: 관심주제 `intrsThemaArray` vs `intrsThemaNmArray`).
공통 이름으로 맞춘 뒤 `구분` 컬럼을 붙여 하나의 DataFrame으로 합칩니다.

In [ ]:
COMMON_COLUMNS = {
    "servId": "서비스ID",
    "servNm": "서비스명",
    "servDgst": "서비스요약",
    "sprtCycNm": "지원주기",
    "srvPvsnNm": "제공유형",
    "inqNum": "조회수",
    "servDtlLink": "상세링크",
}

nat = df_nat.rename(columns={
    **COMMON_COLUMNS,
    "jurMnofNm": "담당기관",
    "lifeArray": "생애주기",
    "trgterIndvdlArray": "가구유형",
    "intrsThemaArray": "관심주제",
})
nat["구분"] = "중앙부처"
nat["시도"] = "전국"

loc = df_loc.rename(columns={
    **COMMON_COLUMNS,
    "bizChrDeptNm": "담당기관",
    "lifeNmArray": "생애주기",
    "trgterIndvdlNmArray": "가구유형",
    "intrsThemaNmArray": "관심주제",
    "ctpvNm": "시도",
    "sggNm": "시군구",
})
loc["구분"] = "지자체"

cols = ["구분", "시도", "시군구", "서비스ID", "서비스명", "담당기관", "서비스요약",
        "생애주기", "가구유형", "관심주제", "지원주기", "제공유형", "조회수", "상세링크"]
df_all = pd.concat([nat, loc], ignore_index=True).reindex(columns=cols)
df_all["조회수"] = pd.to_numeric(df_all["조회수"], errors="coerce")
print(df_all["구분"].value_counts())
df_all.head()

## 4. 간단한 분석 예시

In [ ]:
# 시도별 지자체 서비스 수
df_all[df_all["구분"] == "지자체"]["시도"].value_counts()

In [ ]:
# 조회수 상위 10개 (중앙부처 + 지자체)
df_all.sort_values("조회수", ascending=False)[["구분", "시도", "서비스명", "담당기관", "조회수"]].head(10)

In [ ]:
# 관심주제별 서비스 수 (쉼표로 여러 개가 들어있음)
df_all.assign(관심주제=df_all["관심주제"].str.split(",")).explode("관심주제") \
    .assign(관심주제=lambda d: d["관심주제"].str.strip()) \
    .pivot_table(index="관심주제", columns="구분", values="서비스ID", aggfunc="count", fill_value=0) \
    .sort_values("지자체", ascending=False)

## 5. 검색 / 필터

In [ ]:
# 받아온 데이터에서 키워드로 찾기 (서비스명·요약)
keyword = "청년"
mask = df_all["서비스명"].str.contains(keyword, na=False) | df_all["서비스요약"].str.contains(keyword, na=False)
df_all[mask][["구분", "시도", "시군구", "서비스명", "서비스요약"]]

In [ ]:
# 특정 지역만 보기
df_all[(df_all["시도"] == "서울특별시")][["시군구", "서비스명", "담당기관", "지원주기", "제공유형"]]

## 6. 서비스 상세 조회

In [ ]:
# 중앙부처 서비스 상세
info, lists = fetch_detail(df_nat.loc[0, "servId"], api="중앙부처")
display(pd.Series(info).to_frame("값"))
for name, sub in lists.items():
    print(f"== {name} ({len(sub)}건)")
    display(sub)

In [ ]:
# 지자체 서비스 상세
info, lists = fetch_detail(df_loc.loc[0, "servId"], api="지자체")
display(pd.Series(info).to_frame("값"))
for name, sub in lists.items():
    print(f"== {name} ({len(sub)}건)")
    display(sub)

## 7. 정부24 공공서비스 (odcloud API)

`https://api.odcloud.kr/api/gov24/v3` — JSON으로 응답하며 같은 data.go.kr 인증키를 씁니다.

| 엔드포인트 | 내용 |
|---|---|
| `serviceList` | 공공서비스 목록 (지원대상, 선정기준, 지원내용, 신청방법 등) |
| `serviceDetail` | 공공서비스 상세 (구비서류, 문의처, 법령 등) |
| `supportConditions` | 지원조건 (성별, 연령, 소득, 가구유형 등 Y/N 코드) |

검색 조건은 `cond={"서비스명::LIKE": "청년"}`처럼 넘깁니다.
가능한 조건: `서비스ID::EQ`, `서비스명::LIKE`, `등록일시::GTE/GT/LTE/LT`, `수정일시::GTE/GT/LTE/LT` (상세·지원조건은 `서비스ID::EQ`만)

In [ ]:
GOV24_BASE = "https://api.odcloud.kr/api/gov24/v3"


def fetch_gov24_page(endpoint="serviceList", page=1, per_page=1000, cond=None):
    """정부24 API 한 페이지 → (전체건수, DataFrame)"""
    params = {"serviceKey": SERVICE_KEY, "page": page, "perPage": per_page, "returnType": "JSON"}
    for k, v in (cond or {}).items():
        params[f"cond[{k}]"] = v
    res = requests.get(f"{GOV24_BASE}/{endpoint}", params=params, timeout=60)
    if res.status_code != 200:
        raise RuntimeError(f"API 오류 {res.status_code}: {res.text[:300]}")
    body = res.json()
    return body["matchCount"], pd.DataFrame(body["data"])


def fetch_gov24_all(endpoint="serviceList", per_page=1000, cond=None, sleep=0.2):
    """모든 페이지를 돌며 전체를 DataFrame으로 반환"""
    total, df = fetch_gov24_page(endpoint, 1, per_page, cond)
    frames = [df]
    pages = (total + per_page - 1) // per_page
    for page in range(2, pages + 1):
        time.sleep(sleep)
        _, df = fetch_gov24_page(endpoint, page, per_page, cond)
        frames.append(df)
    result = pd.concat(frames, ignore_index=True)
    print(f"[정부24 {endpoint}] 전체 {total}건 중 {len(result)}건 수집")
    return result


def clean_text_columns(df):
    # \r\n → \n 정리
    obj = df.select_dtypes("object").columns
    df[obj] = df[obj].apply(lambda s: s.str.replace("\r\n", "\n", regex=False))
    return df

### 7-1. 공공서비스 목록

In [ ]:
df_gov = clean_text_columns(fetch_gov24_all("serviceList"))
df_gov["조회수"] = pd.to_numeric(df_gov["조회수"], errors="coerce")
df_gov.head()

In [ ]:
# 서비스분야 / 소관기관유형별 건수
display(df_gov["서비스분야"].value_counts())
df_gov["소관기관유형"].value_counts()

In [ ]:
# API에서 바로 검색: 서비스명에 '청년'이 들어간 서비스
df_gov_youth = fetch_gov24_all("serviceList", cond={"서비스명::LIKE": "청년"})
df_gov_youth[["서비스ID", "서비스명", "소관기관명", "지원유형", "서비스목적요약"]]

### 7-2. 공공서비스 상세

In [ ]:
# 한 건만 조회
_, detail = fetch_gov24_page("serviceDetail", cond={"서비스ID::EQ": df_gov.loc[0, "서비스ID"]})
detail.T.rename(columns={0: "값"})

In [ ]:
# 전체 상세 (약 1~2분 소요)
df_gov_detail = clean_text_columns(fetch_gov24_all("serviceDetail"))
df_gov_detail.head()

### 7-3. 지원조건

`JA0101` 같은 코드 컬럼을 사람이 읽을 수 있는 이름으로 바꿉니다.
값이 `Y`면 해당 조건의 사람이 지원 대상이라는 뜻입니다.

In [ ]:
SUPPORT_CODES = {
    "JA0101": "남성", "JA0102": "여성",
    "JA0110": "대상연령_시작", "JA0111": "대상연령_종료",
    "JA0201": "중위소득_0~50%", "JA0202": "중위소득_51~75%", "JA0203": "중위소득_76~100%",
    "JA0204": "중위소득_101~200%", "JA0205": "중위소득_200%초과",
    "JA0301": "예비부모/난임", "JA0302": "임산부", "JA0303": "출산/입양",
    "JA0313": "농업인", "JA0314": "어업인", "JA0315": "축산업인", "JA0316": "임업인",
    "JA0317": "초등학생", "JA0318": "중학생", "JA0319": "고등학생", "JA0320": "대학생/대학원생",
    "JA0322": "대상특성_해당사항없음",
    "JA0326": "근로자/직장인", "JA0327": "구직자/실업자",
    "JA0328": "장애인", "JA0329": "국가보훈대상자", "JA0330": "질병/질환자",
    "JA0401": "다문화가족", "JA0402": "북한이탈주민", "JA0403": "한부모/조손가정",
    "JA0404": "1인가구", "JA0410": "가구유형_해당사항없음", "JA0411": "다자녀가구",
    "JA0412": "무주택세대", "JA0413": "신규전입", "JA0414": "확대가족",
    "JA1101": "소상공인_예비창업자", "JA1102": "소상공인_영업중", "JA1103": "소상공인_생계곤란/폐업예정",
    "JA1201": "소상공인_음식점업", "JA1202": "소상공인_제조업", "JA1299": "소상공인_기타업종",
    "JA2101": "법인_중소기업", "JA2102": "법인_사회복지시설", "JA2103": "법인_기관/단체",
    "JA2201": "법인_제조업", "JA2202": "법인_농업·임업·어업", "JA2203": "법인_정보통신업",
    "JA2299": "법인_기타업종",
}

df_gov_cond = fetch_gov24_all("supportConditions").rename(columns=SUPPORT_CODES)
df_gov_cond.head()

In [ ]:
# 예시: 1인가구 + 구직자/실업자 + 25세가 대상인 서비스 찾기
age = 25
cond = df_gov_cond
start = pd.to_numeric(cond["대상연령_시작"], errors="coerce").fillna(0)
end = pd.to_numeric(cond["대상연령_종료"], errors="coerce").fillna(200)
matched = cond[
    (cond["1인가구"] == "Y")
    & (cond["구직자/실업자"] == "Y")
    & (start <= age) & (age <= end)
]
df_gov.merge(matched[["서비스ID"]], on="서비스ID")[["서비스명", "소관기관명", "지원유형", "서비스목적요약", "상세조회URL"]]

## 8. CSV로 저장
엑셀에서 한글이 깨지지 않도록 `utf-8-sig`로 저장합니다.

In [ ]:
df_nat.to_csv("welfare_national.csv", index=False, encoding="utf-8-sig")
df_loc.to_csv("welfare_local.csv", index=False, encoding="utf-8-sig")
df_all.to_csv("welfare_all.csv", index=False, encoding="utf-8-sig")
df_gov.to_csv("gov24_service_list.csv", index=False, encoding="utf-8-sig")
df_gov_detail.to_csv("gov24_service_detail.csv", index=False, encoding="utf-8-sig")
df_gov_cond.to_csv("gov24_support_conditions.csv", index=False, encoding="utf-8-sig")